# Initialize spark session

In [18]:
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .config("spark.driver.memory", "5g") \
    .config("spark.executor.memory", "5g") \
    .appName("Open food facts") \
    .getOrCreate()

# Load the data
data = spark.read.csv(
    "/Users/amine/Desktop/Spark-Recommendation-System/data/final_data_cleaned.csv",
    header=True,
)

In [19]:
data.show(5, truncate=False)

+-------------+---------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------+-----------------------------------------------------------------------+----------+-----------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [20]:
data.printSchema()

root
 |-- code: string (nullable = true)
 |-- url: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- generic_name: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- brands: string (nullable = true)
 |-- categories_tags: string (nullable = true)
 |-- categories_en: string (nullable = true)
 |-- ingredients_tags: string (nullable = true)
 |-- ingredients_analysis_tags: string (nullable = true)
 |-- additives_tags: string (nullable = true)
 |-- nutriscore_grade: string (nullable = true)
 |-- nova_group: string (nullable = true)
 |-- product_quantity: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- custom_completeness: string (nullable = true)



# Create a column with all words related to product

In [21]:
from pyspark.sql.functions import expr, array, flatten

df_with_tokens = data.withColumn(
    "categories_tokens",
    expr("""
        transform(
            filter(split(categories_tags, ','), x -> startswith(x, 'en:')),
            x -> replace(x, 'en:', '')
        )
    """)
).withColumn(
    "main_category_tokens",
    expr("""
        transform(
            filter(split(main_category, ','), x -> startswith(x, 'en:')),
            x -> replace(x, 'en:', '')
        )
    """)
)

df_with_tokens = df_with_tokens.withColumn(
    "all_tokens",
    flatten(array(
        "categories_tokens",
        "main_category_tokens",
    ))
)

df_with_tokens = df_with_tokens.withColumn("all_tokens", expr("array_distinct(all_tokens)"))

df_with_tokens.select("all_tokens").show(truncate=False)


+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|all_tokens                                                                                                                                                                                                                                                                            |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[dairies, fermented-foods, fermented-milk-products, desserts, dairy-desserts, fermented-dairy-desserts, plain-fermented-dairy-desserts, skyrs, plain-skyrs] 

In [22]:
import nltk
nltk.data.path.append('/Users/amine/Desktop/Spark-Recommendation-System/nltk_data')

In [23]:
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))


def remove_hyphens(tokens):
    return [token.replace("-", " ") for token in tokens]


def explode_phrases(tokens):
    words = []
    for token in tokens:
        words.extend(token.lower().split())
    return list(set(words))


def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]



remove_hyphens_udf = udf(remove_hyphens, ArrayType(StringType()))
explode_phrases_udf = udf(explode_phrases, ArrayType(StringType()))
remove_stopwords_udf = udf(remove_stopwords, ArrayType(StringType()))


df_tokens_cleaned = df_with_tokens.withColumn("all_tokens", remove_hyphens_udf("all_tokens"))
tokens_df = df_tokens_cleaned.withColumn("clean_tokens", explode_phrases_udf("all_tokens"))
final_df = tokens_df.withColumn("tokens", remove_stopwords_udf("clean_tokens"))



In [24]:
final_df.printSchema()

root
 |-- code: string (nullable = true)
 |-- url: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- generic_name: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- brands: string (nullable = true)
 |-- categories_tags: string (nullable = true)
 |-- categories_en: string (nullable = true)
 |-- ingredients_tags: string (nullable = true)
 |-- ingredients_analysis_tags: string (nullable = true)
 |-- additives_tags: string (nullable = true)
 |-- nutriscore_grade: string (nullable = true)
 |-- nova_group: string (nullable = true)
 |-- product_quantity: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- custom_completeness: string (nullable = true)
 |-- categories_tokens: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- main_category_tokens: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- all_tokens: array (nullable = true)
 |

In [25]:
final_df.select("tokens").show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------+
|tokens                                                                                                                                          |
+------------------------------------------------------------------------------------------------------------------------------------------------+
|[dairy, desserts, plain, dairies, products, skyrs, fermented, milk, foods]                                                                      |
|[seafood, fishes, farming, products, salmons, smoked, fatty]                                                                                    |
|[condiments, dessert, sauces]                                                                                                                   |
|[preparations, hot, plant, coffees, beverage, based, beverages, decaffeinated, instant, foods]                       

# Recommendation System with aggregated vectors

## Trained Word2vec

In [35]:
from pyspark.ml.feature import Word2Vec
from pyspark.ml.feature import Word2VecModel


word2vec = Word2Vec(
    inputCol="tokens",     
    outputCol="embedding", 
    vectorSize=100,        
    minCount=1             
)

model = word2vec.fit(final_df)

model.save("/Users/amine/Desktop/Spark-Recommendation-System/models/custom_word2vec_model")

In [36]:
loaded_model = Word2VecModel.load("/Users/amine/Desktop/Spark-Recommendation-System/models/custom_word2vec_model")

final_df.show(5)

+-------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+-------------------------+--------------------+----------------+----------+----------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|         code|                 url|        product_name|        generic_name|  quantity|              brands|     categories_tags|       categories_en|    ingredients_tags|ingredients_analysis_tags|      additives_tags|nutriscore_grade|nova_group|product_quantity|       main_category|           image_url|custom_completeness|   categories_tokens|main_category_tokens|          all_tokens|        clean_tokens|              tokens|
+-------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+---

In [27]:
final_df_with_embeddings_custom_word2vec = loaded_model.transform(final_df)

In [28]:
import numpy as np

user_word = "potato"

try:
    user_vec_row = loaded_model.getVectors().filter(f"word = '{user_word}'").collect()[0]
    user_vec_np = np.array(user_vec_row['vector'])
except IndexError:
    raise ValueError(f"The word '{user_word}' was not found in the vocabulary.")


In [29]:
user_vec_broadcast = spark.sparkContext.broadcast(user_vec_np)

from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def cosine_similarity(vec):
    import numpy as np
    vec = np.array(vec)
    user_vec = user_vec_broadcast.value
    dot = np.dot(vec, user_vec)
    norm1 = np.linalg.norm(vec)
    norm2 = np.linalg.norm(user_vec)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(dot / (norm1 * norm2))

cosine_similarity_udf = udf(cosine_similarity, DoubleType())


In [30]:
from pyspark.sql.functions import col

df_with_similarity = final_df_with_embeddings_custom_word2vec.withColumn("similarity", cosine_similarity_udf(col("embedding")))

In [31]:
# Show top 10 similar items
top_similar = df_with_similarity.orderBy(col("similarity").desc()).limit(10)
top_similar.show(truncate=False)

+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------+----------------------------------------------------------+--------+----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------

In [38]:


final_df_with_embeddings_custom_word2vec.write.mode("overwrite").parquet("/Users/amine/Desktop/Spark-Recommendation-System/data/final_data_with_embeddings.parquet")

## Pre trained Word2vec

In [32]:
# from gensim.models import keyedvectors

# model = keyedvectors.KeyedVectors.load_word2vec_format(
#     "/Users/amine/Downloads/GoogleNews-vectors-negative300.bin.gz",
#     binary=True
# )

# print(f"Model loaded with success.")

In [33]:
# import gensim.downloader

# glove_vectors = gensim.downloader.load('glove-wiki-gigaword-100')
